In [1]:
import sqlite3
import threading
import uvicorn
import nest_asyncio
import pandas as pd
import numpy as np
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DB_NAME = "cinema.db"

class RecommenderService:
    def __init__(self, db_path=DB_NAME):
        self.db_path = db_path
        self.refresh()

    def get_conn(self):
        return sqlite3.connect(self.db_path)

    def refresh(self):
        conn = self.get_conn()
        self.movies_df = pd.read_sql_query("SELECT * FROM movies", conn)
        self.ratings_df = pd.read_sql_query("SELECT * FROM ratings", conn)
        conn.close()

        # Коллаборативная матрица
        merged = self.ratings_df.merge(self.movies_df[['id', 'title']], left_on='movie_id', right_on='id')
        self.pivot = merged.pivot_table(index='user_id', columns='title', values='rating').fillna(0)

        self.item_matrix = csr_matrix(self.pivot.values).T
        self.model_item_knn = NearestNeighbors(metric='cosine', algorithm='brute')
        self.model_item_knn.fit(self.item_matrix)
        self.titles = list(self.pivot.columns)

        self.user_matrix = csr_matrix(self.pivot.values)
        self.model_user_knn = NearestNeighbors(metric='cosine', algorithm='brute')
        self.model_user_knn.fit(self.user_matrix)

        # объединяем НАЗВАНИЕ + ЖАНРЫ + ОПИСАНИЕ
        self.movies_df['content'] = (
            self.movies_df['title'].fillna('') + ' ' +
            self.movies_df['genres'].fillna('') + ' ' + 
            self.movies_df['description'].fillna('')
        )
        # векторизуем общий текст
        tfidf = TfidfVectorizer(max_features=5000)
        self.tfidf_matrix = tfidf.fit_transform(self.movies_df['content'])
        self.content_sim = cosine_similarity(self.tfidf_matrix, self.tfidf_matrix)

    def get_top_10(self):
        stats = self.ratings_df.groupby('movie_id').agg(
            v=('rating', 'count'),
            R=('rating', 'mean')
        ).reset_index()

        c = stats['R'].mean()
        m = stats['v'].quantile(0.70)
        stats['w_score'] = ((stats['v'] * stats['R']) + (m * c)) / (stats['v'] + m)
        top = stats.sort_values(by='w_score', ascending=False).head(10)
        
        res = top.merge(self.movies_df, left_on='movie_id', right_on='id')
        return res[['id', 'title', 'year', 'genres', 'poster_url', 'rating_kp', 'w_score']].to_dict(orient='records')

    def recommend_by_movie(self, title, limit=10):
        if title not in self.titles:
            return []
        idx = self.titles.index(title)
        distances, indices = self.model_item_knn.kneighbors(self.item_matrix[idx], n_neighbors=limit + 1)
        
        res = []
        for i, dist in zip(indices[0][1:], distances[0][1:]):
            m_title = self.titles[i]
            meta = self.movies_df[self.movies_df['title'] == m_title].iloc[0]
            res.append({
                "id": int(meta['id']),
                "title": m_title,
                "year": int(meta['year']),
                "genres": meta['genres'],
                "poster_url": meta['poster_url'],
                "score": round(float(1 - dist), 4)
            })
        return res

    # Рекомендации по контенту
    def recommend_by_content(self, title, limit=10):
        matches = self.movies_df[self.movies_df['title'].str.lower() == title.lower()]
        if matches.empty:
            return []
        idx = matches.index[0]
        scores = sorted(list(enumerate(self.content_sim[idx])), key=lambda x: x[1], reverse=True)[1:limit + 1]
        
        res = []
        for i, score in scores:
            m = self.movies_df.iloc[i]
            res.append({
                "id": int(m['id']),
                "title": m['title'],
                "year": int(m['year']),
                "genres": m['genres'],
                "poster_url": m['poster_url'],
                "score": round(float(score), 4)
            })
        return res

    # Персональные рекомендации для любого пользователя
    def recommend_personal(self, user_id, limit=10):
        user_ratings_db = self.ratings_df[self.ratings_df['user_id'] == user_id]
        
        # Если у пользователя нет оценки - отдаем Топ-10
        if user_ratings_db.empty:
            return self.get_top_10()[:limit]

        # Если пользователь есть в матрице пользователей
        if user_id in self.pivot.index:
            u_idx = self.pivot.index.get_loc(user_id)
            n_neighbors = min(15, len(self.pivot))
            distances, indices = self.model_user_knn.kneighbors(
                self.user_matrix[u_idx], 
                n_neighbors=n_neighbors
            )
            
            weights = 1.0 / (distances[0][1:] + 1e-9)
            similar_ratings = self.pivot.iloc[indices[0][1:]]
            weighted_scores = similar_ratings.T.dot(weights) / weights.sum()

            target_ratings = self.pivot.iloc[u_idx]
            unseen = target_ratings[target_ratings == 0].index
            candidates = weighted_scores.loc[unseen].sort_values(ascending=False).head(limit)

            res = []
            for title, score in candidates.items():
                meta = self.movies_df[self.movies_df['title'] == title].iloc[0]
                res.append({
                    "id": int(meta['id']),
                    "title": title,
                    "year": int(meta['year']),
                    "genres": meta['genres'],
                    "poster_url": meta['poster_url'],
                    "score": round(float(score), 3)
                })
            return res

        # Если пользователь только что создан и имеет лишь пару оценок
        fav_movie_id = user_ratings_db.sort_values(by='rating', ascending=False).iloc[0]['movie_id']
        fav_movie_title = self.movies_df[self.movies_df['id'] == fav_movie_id]['title'].values[0]
        return self.recommend_by_content(fav_movie_title, limit=limit)

    def add_rating(self, user_id, movie_id, rating):
        conn = self.get_conn()
        cur = conn.cursor()
        cur.execute("INSERT INTO ratings (user_id, movie_id, rating) VALUES (?, ?, ?)", (user_id, movie_id, rating))
        conn.commit()
        conn.close()
        self.refresh()

app = FastAPI(title="Movie Recommendation API", version="1.1")
service = RecommenderService()

class RatingRequest(BaseModel):
    user_id: int
    movie_id: int
    rating: float

@app.get("/recommend/top")
def get_popular():
    return service.get_top_10()

@app.get("/recommend/by-movie")
def get_by_movie(title: str = Query(...)):
    results = service.recommend_by_movie(title)
    if not results:
        raise HTTPException(status_code=404, detail="Фильм не найден в матрице")
    return results

@app.get("/recommend/by-content")
def get_by_content(title: str = Query(...)):
    results = service.recommend_by_content(title)
    if not results:
        raise HTTPException(status_code=404, detail="Фильм не найден в каталоге")
    return results

@app.get("/recommend/personal/{user_id}")
def get_personal(user_id: int):
    return service.recommend_personal(user_id)

@app.get("/movies")
def get_all_movies():
    return service.movies_df[['id', 'title', 'year', 'genres', 'poster_url']].to_dict(orient='records')

@app.post("/rate")
def save_rating(data: RatingRequest):
    if not (1.0 <= data.rating <= 10.0):
        raise HTTPException(status_code=400, detail="Оценка должна быть от 1 до 10")
    service.add_rating(data.user_id, data.movie_id, data.rating)
    return {"status": "success", "message": "Оценка сохранена и модели переобучены"}

@app.get("/help")
def get_help():
    return {
        "endpoints": {
            "GET /recommend/top": "Топ-10 фильмов по формуле взвешенного рейтинга",
            "GET /recommend/by-movie?title=<название>": "Коллаборативная фильтрация по похожим фильмам",
            "GET /recommend/by-content?title=<название>": "10 рекомендаций по объединенному тексту (Название + Жанры + Описание через TF-IDF)",
            "GET /recommend/personal/{user_id}": "Персональные рекомендации для пользователя",
            "GET /movies": "Каталог доступных фильмов",
            "POST /rate": "Сохранить оценку фильма пользователем (JSON: user_id, movie_id, rating)",
            "GET /docs": "Интерактивная Swagger-документация"
        }
    }

def start_server():
    nest_asyncio.apply()
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

threading.Thread(target=start_server, daemon=True).start()
print("Сервер API обновлен и запущен на http://127.0.0.1:8000")

Сервер API обновлен и запущен на http://127.0.0.1:8000


INFO:     Started server process [16928]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:53917 - "GET /movies HTTP/1.1" 200 OK
INFO:     127.0.0.1:53798 - "GET /help HTTP/1.1" 200 OK
INFO:     127.0.0.1:60598 - "GET /movies HTTP/1.1" 200 OK
INFO:     127.0.0.1:60599 - "GET /recommend/by-movie?title=%D0%9C%D0%B0%D1%82%D1%80%D0%B8%D1%86%D0%B0 HTTP/1.1" 200 OK
INFO:     127.0.0.1:60600 - "GET /help HTTP/1.1" 200 OK
INFO:     127.0.0.1:58897 - "GET /help HTTP/1.1" 200 OK
INFO:     127.0.0.1:58898 - "GET /recommend/by-movie?title=%D0%9F%D0%B8%D1%80%D0%B0%D1%82%D1%8B+%D0%9A%D0%B0%D1%80%D0%B8%D0%B1%D1%81%D0%BA%D0%BE%D0%B3%D0%BE+%D0%BC%D0%BE%D1%80%D1%8F%3A+%D0%9D%D0%B0+%D0%BA%D1%80%D0%B0%D1%8E+%D1%81%D0%B2%D0%B5%D1%82%D0%B0 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58899 - "GET /help HTTP/1.1" 200 OK
INFO:     127.0.0.1:61886 - "GET /movies HTTP/1.1" 200 OK
INFO:     127.0.0.1:61887 - "GET /recommend/by-movie?title=%D0%9F%D0%B8%D1%80%D0%B0%D1%82%D1%8B+%D0%9A%D0%B0%D1%80%D0%B8%D0%B1%D1%81%D0%BA%D0%BE%D0%B3%D0%BE+%D0%BC%D0%BE%D1%80%D1%8F%3A+%D0%9D%D0%B0+%D0%BA%D1%80%D